# LAB 6 &ndash; Improving Our Nodes (Consensus, Part 2)

## Done by,
*   Mishael Natth VISWANATHAN
*   Nathan BOUDRY
*   Téophile PRADALIER-LEFEBVRE
*   Aaron Rohit JEROLD


---

## Overview

In this lab, we build upon the peer-to-peer `Node` from Chapter 4 (*Decentralization*) and the cryptographic validation and blockchain structures from Chapter 5 (*Consensus, Part 1*). We implement realistic decentralized consensus and node management as outlined in the lecture:

1. **Max Peers Parameter (`max_peers`)**: Restrict peer connections to a realistic bound to prevent resource exhaustion and Sybil vulnerability.
2. **Liveness & Failure Detection (`check_alive_peers` / `ping`)**: Detect unreachable/offline nodes and prune them from the available peers table.
3. **Chain Inquiry upon Joining (`request_chain_info`)**: When a new node joins the network, query peers for the lengths and terminal hashes of their chains.
4. **Longest-Chain Rule & Clever Synchronization**: When chain information is received, adopt longer valid chains while **minimizing the number of blocks transmitted** (delta synchronization).
5. **Consensus Propagation (`broadcast_chain_update`)**: Whenever the local chain is updated (mined or synced), broadcast the new length and terminal hash to all peers.


## 0. Setup and Dependencies

We re-use `JCrypto` and `JChain` developed in the previous courses.

In [1]:
import socket
import threading
import sys
import json
import time
import os

# Import modules from previous courses
import JCrypto
import JChain

print("JCrypto and JChain successfully loaded.")


JChain v0.5 loaded - now with fuzz parameter in blocks
JCrypto and JChain successfully loaded.


## Block Serialization Helpers

To transmit `Block` objects across TCP sockets using JSON messages, we define serialization and deserialization functions (`block_to_dict` and `dict_to_block`) and attach them to `JChain.Block`.

In [2]:
def block_to_dict(block):
    """Serializes a JChain.Block into a JSON-compatible dictionary."""
    return {
        "content": block.content,
        "prev_hash": block.prev_hash,
        "fuzz": block.fuzz,
        "public_key": block.public_key,
        "signature": block.signature
    }

def dict_to_block(d):
    """Reconstructs a JChain.Block from a dictionary."""
    return JChain.Block(
        content=d.get("content"),
        prev_hash=d.get("prev_hash"),
        fuzz=d.get("fuzz"),
        public_key=d.get("public_key"),
        signature=d.get("signature")
    )

# Attach helpers to JChain.Block
JChain.Block.to_dict = block_to_dict
JChain.Block.from_dict = staticmethod(dict_to_block)

# Quick verification test
sample_b = JChain.Block("Genesis operation", prev_hash=None, fuzz=12345)
sample_dict = sample_b.to_dict()
reconstructed_b = JChain.Block.from_dict(sample_dict)
assert sample_b.hash() == reconstructed_b.hash()
print(f"[OK] Serialization verified. Block hash: {hex(sample_b.hash())}")


[OK] Serialization verified. Block hash: 0xf765a5c


## The Improved `Node` Class

Below is the complete implementation of `Node`, integrating all 5 exercises:
- **Exercise 1**: `max_peers` parameter and peer connection limiter in `add_peer()`.
- **Exercise 2**: `check_alive_peers()` command to ping known peers and prune unreachable nodes.
- **Exercise 3**: `request_chain_info()` invoked after joining to ask peers for chain lengths and terminal hashes.
- **Exercise 4**: Handlers for `CHAIN_INFO`, `GET_BLOCKS`, and `BLOCKS` implementing the longest-chain rule with **minimal block transmission** (requesting only the delta blocks from `current_length` when common history matches).
- **Exercise 5**: `broadcast_chain_update()` invoked whenever the chain is updated (locally mined or peer-synced).


In [3]:
class Node:
    def __init__(self, host, port, peers=None, max_peers=8, chain=None, initial_state=None, private_key=None, target=None):
        """
        Exercise 1:
        Implements a `max_peers` parameter and only allows connection to up to this number of peers.
        """
        self.host = host
        self.port = int(port)
        self.address = (host, self.port)
        self.max_peers = int(max_peers)
        
        # Thread safety lock for peer list and blockchain
        self.lock = threading.Lock()
        
        # Filter self-address and truncate initial peers to max_peers
        raw_peers = peers if peers is not None else []
        filtered_peers = []
        for p in raw_peers:
            peer_tuple = (p[0], int(p[1]))
            if peer_tuple != self.address and peer_tuple not in filtered_peers:
                filtered_peers.append(peer_tuple)
        self.peers = filtered_peers[:self.max_peers]

        # Blockchain state
        self.chain = chain if chain is not None else JChain.Blockchain()
        self.initial_state = initial_state if initial_state is not None else JChain.State({})
        self.private_key = private_key
        self.public_key = JCrypto.ExtractPublicKey(private_key) if private_key is not None else None
        self.target = target

        # Network lifecycle
        self.running = False
        self.server_socket = None

    # ---------------------------------------------------------
    # Network Lifecycle (Start, Listen, Stop)
    # ---------------------------------------------------------

    def start(self):
        """Starts listening, announces HELLO, discovers peers, and asks for chain info (Exercise 3)."""
        self.running = True
        threading.Thread(target=self.listen, daemon=True).start()
        time.sleep(0.05)

        self.broadcast({"type": "HELLO", "from": self.address})
        self.discover_peers()

        # Exercise 3: After joining the network, ask peers for lengths & terminal hashes
        self.request_chain_info()

    def listen(self):
        """TCP server loop listening for incoming peer messages."""
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind(self.address)
            s.listen()
            self.server_socket = s
            print(f"[+] Node listening on {self.address} (max_peers={self.max_peers})")

            while self.running:
                try:
                    conn, _ = s.accept()
                    data = b""
                    while True:
                        chunk = conn.recv(8192)
                        if not chunk:
                            break
                        data += chunk
                    conn.close()

                    if data:
                        try:
                            msg = json.loads(data.decode("utf-8"))
                            self.handle_message(msg)
                        except json.JSONDecodeError as err:
                            print(f"[-] JSON decode error on {self.port}: {err}")
                except Exception:
                    if not self.running:
                        break
        except Exception as e:
            if self.running:
                print(f"[-] Socket error on {self.port}: {e}")
        finally:
            try:
                s.close()
            except Exception:
                pass

    def stop(self):
        """Gracefully shuts down the node listener socket."""
        self.running = False
        if self.server_socket:
            try:
                self.server_socket.close()
            except Exception:
                pass
        print(f"[*] Node on {self.port} stopped.")

    # ---------------------------------------------------------
    # Basic Communication
    # ---------------------------------------------------------

    def send(self, peer, message, timeout=2.0):
        """Sends a JSON-encoded message to a target peer."""
        try:
            peer_tuple = (peer[0], int(peer[1]))
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.settimeout(timeout)
            s.connect(peer_tuple)
            payload = json.dumps(message).encode("utf-8")
            s.sendall(payload)
            s.shutdown(socket.SHUT_WR)
            s.close()
            return True
        except Exception:
            return False

    def broadcast(self, message):
        """Sends a message to all currently connected peers."""
        with self.lock:
            peers_snapshot = list(self.peers)
        for peer in peers_snapshot:
            self.send(peer, message)

    # ---------------------------------------------------------
    # Exercise 1: Peer Management with max_peers
    # ---------------------------------------------------------

    def add_peer(self, peer):
        """
        Exercise 1:
        Only allow connection to up to max_peers peers.
        Returns True if peer was added, False otherwise.
        """
        peer_tuple = (peer[0], int(peer[1]))
        if peer_tuple == self.address:
            return False

        with self.lock:
            if peer_tuple in self.peers:
                return False
            if len(self.peers) >= self.max_peers:
                print(f"[-] [max_peers={self.max_peers}] Limit reached. Cannot add peer {peer_tuple}")
                return False
            self.peers.append(peer_tuple)
            print(f"[+] Added peer {peer_tuple} ({len(self.peers)}/{self.max_peers} connected)")
            return True

    def remove_peer(self, peer):
        """Removes a peer from the known peers list."""
        peer_tuple = (peer[0], int(peer[1]))
        with self.lock:
            if peer_tuple in self.peers:
                self.peers.remove(peer_tuple)
                return True
        return False

    def discover_peers(self, peer=None):
        """Sends GET_PEERS request to a specific peer or broadcasts to all peers."""
        msg = {"type": "GET_PEERS", "from": self.address}
        if peer:
            self.send(peer, msg)
        else:
            self.broadcast(msg)

    # ---------------------------------------------------------
    # Exercise 2: Alive Check (Ping Unreachable Nodes)
    # ---------------------------------------------------------

    def check_alive_peers(self, timeout=1.0):
        """
        Exercise 2:
        Some known nodes may become unreachable.
        Asks all known peers if they are still alive (via PING/connect probe)
        and updates the list of available known peers accordingly.
        """
        print(f"[*] Checking liveness of {len(self.peers)} known peer(s)...")
        with self.lock:
            peers_to_check = list(self.peers)

        alive_peers = []
        unreachable_peers = []

        for peer in peers_to_check:
            is_alive = False
            try:
                s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                s.settimeout(timeout)
                s.connect(peer)
                s.sendall(json.dumps({"type": "PING", "from": self.address}).encode("utf-8"))
                s.shutdown(socket.SHUT_WR)
                s.close()
                is_alive = True
            except (socket.timeout, ConnectionRefusedError, OSError):
                is_alive = False

            if is_alive:
                alive_peers.append(peer)
                print(f"    [OK] Peer {peer} is alive.")
            else:
                unreachable_peers.append(peer)
                print(f"    [X]  Peer {peer} is unreachable! Pruning from peers list.")

        with self.lock:
            self.peers = alive_peers

        print(f"[+] Liveness check complete: {len(self.peers)} available, {len(unreachable_peers)} removed.")
        return alive_peers, unreachable_peers

    def ping_peers(self, timeout=1.0):
        """Alias for check_alive_peers."""
        return self.check_alive_peers(timeout=timeout)

    # ---------------------------------------------------------
    # Exercise 3: Chain Information Request on Join
    # ---------------------------------------------------------

    def request_chain_info(self, peer=None):
        """
        Exercise 3:
        Ask peers (or a specific peer) for the lengths and terminal hashes of their chains.
        """
        msg = {"type": "GET_CHAIN_INFO", "from": self.address}
        if peer:
            print(f"[*] Requesting chain info from {peer}")
            self.send(peer, msg)
        else:
            print(f"[*] Asking all known peers for their chain lengths & terminal hashes...")
            self.broadcast(msg)

    # ---------------------------------------------------------
    # Exercise 5: Broadcast Updated Chain
    # ---------------------------------------------------------

    def broadcast_chain_update(self):
        """
        Exercise 5:
        Whenever the current chain is updated, the current length and terminal
        hash of the chain is broadcasted.
        """
        info = {
            "type": "CHAIN_INFO",
            "from": self.address,
            "length": len(self.chain),
            "hash": self.chain.hash()
        }
        print(f"[!] Broadcasting CHAIN_INFO: length={len(self.chain)}, terminal_hash={self.chain.hash()}")
        self.broadcast(info)

    def add_block(self, content, fuzz=None):
        """
        Appends a new block to the local blockchain and broadcasts the new chain info (Exercise 5).
        """
        prev_hash = self.chain.hash() if len(self.chain) > 0 else None
        if self.private_key is not None:
            new_block = JChain.Block.create(content, self.private_key, prev_hash=prev_hash, fuzz=fuzz)
        else:
            new_block = JChain.Block(content, prev_hash=prev_hash, fuzz=fuzz)

        with self.lock:
            self.chain.append(new_block)
            print(f"[+] New block added locally: length={len(self.chain)}, hash={self.chain.hash()}")

        # Exercise 5: Broadcast length and terminal hash whenever chain is updated
        self.broadcast_chain_update()
        return new_block

    # ---------------------------------------------------------
    # Exercise 4: Chain Validation & Clever Sync
    # ---------------------------------------------------------

    def is_chain_valid(self, chain):
        """Validates hash linkage, signatures, and optional state rules."""
        if not chain.check_hashes():
            return False
        if self.initial_state:
            try:
                return chain.is_valid(self.initial_state, target=self.target)
            except Exception:
                pass
        for b in chain:
            if getattr(b, "signature", None) is not None and getattr(b, "public_key", None) is not None:
                if not b.is_signature_valid():
                    return False
        return True

    # ---------------------------------------------------------
    # Message Router & Handlers
    # ---------------------------------------------------------

    def handle_message(self, msg):
        """Routes incoming messages by type."""
        t = msg.get("type")

        # ------------------- HELLO -------------------
        if t == "HELLO":
            peer = tuple(msg["from"])
            print(f"[+] HELLO received from {peer}")
            added = self.add_peer(peer)
            if added:
                self.send(peer, {"type": "GET_CHAIN_INFO", "from": self.address})

        # ------------------- GET_PEERS -------------------
        elif t == "GET_PEERS":
            sender = tuple(msg["from"])
            self.add_peer(sender)
            with self.lock:
                peers_list = list(self.peers)
            self.send(sender, {"type": "PEERS", "from": self.address, "peers": peers_list})

        # ------------------- PEERS -------------------
        elif t == "PEERS":
            for p in msg.get("peers", []):
                peer = tuple(p)
                if peer != self.address and peer not in self.peers:
                    if len(self.peers) < self.max_peers:
                        if self.add_peer(peer):
                            self.send(peer, {"type": "HELLO", "from": self.address})
                            self.discover_peers(peer)
                            self.request_chain_info(peer)
                    else:
                        print(f"[-] [max_peers={self.max_peers}] Skipping peer {peer}")
                        break

        # ------------------- PING / PONG (Exercise 2) -------------------
        elif t == "PING":
            sender = tuple(msg["from"])
            self.send(sender, {"type": "PONG", "from": self.address})

        elif t == "PONG":
            sender = tuple(msg["from"])
            with self.lock:
                self.alive_responses.add(sender)

        # ------------------- GET_CHAIN_INFO (Exercise 3) -------------------
        elif t == "GET_CHAIN_INFO":
            sender = tuple(msg["from"])
            self.send(sender, {
                "type": "CHAIN_INFO",
                "from": self.address,
                "length": len(self.chain),
                "hash": self.chain.hash()
            })

        # ------------------- CHAIN_INFO (Exercise 4) -------------------
        elif t == "CHAIN_INFO":
            sender = tuple(msg["from"])
            peer_length = msg.get("length", 0)
            peer_hash = msg.get("hash")
            current_length = len(self.chain)
            current_hash = self.chain.hash()

            print(f"[i] CHAIN_INFO from {sender}: length={peer_length}, hash={peer_hash}")

            # Longest chain rule: between two valid chains, always choose the longest one
            if peer_length > current_length:
                print(f"[*] Peer {sender} has a longer chain ({peer_length} > {current_length}). Requesting update...")
                # Clever synchronization: minimize number of blocks transmitted!
                if current_length == 0:
                    self.send(sender, {
                        "type": "GET_BLOCKS",
                        "from": self.address,
                        "start_index": 0,
                        "last_hash": None
                    })
                else:
                    print(f"[*] Clever sync: asking only for missing blocks from index {current_length} (saving {current_length} blocks)!")
                    self.send(sender, {
                        "type": "GET_BLOCKS",
                        "from": self.address,
                        "start_index": current_length,
                        "last_hash": current_hash
                    })
            elif peer_length == current_length and peer_hash == current_hash:
                print(f"[=] Chain is synchronized with {sender} (length={current_length}).")
            else:
                print(f"[-] Peer chain ({peer_length}) is not longer than ours ({current_length}). No update needed.")

        # ------------------- GET_BLOCKS (Exercise 4) -------------------
        elif t == "GET_BLOCKS":
            sender = tuple(msg["from"])
            start_index = msg.get("start_index", 0)
            last_hash = msg.get("last_hash")

            with self.lock:
                total_len = len(self.chain)
                if start_index == 0:
                    blocks_to_send = list(self.chain)
                    effective_start = 0
                elif 0 < start_index <= total_len:
                    if self.chain[start_index - 1].hash() == last_hash:
                        # Common history verified: send ONLY the missing blocks!
                        blocks_to_send = list(self.chain[start_index:])
                        effective_start = start_index
                    else:
                        print(f"[!] Fork detected with {sender}. Sending full chain for conflict resolution.")
                        blocks_to_send = list(self.chain)
                        effective_start = 0
                else:
                    blocks_to_send = []
                    effective_start = start_index

            serialized = [block_to_dict(b) for b in blocks_to_send]
            print(f"[>] Transmitting {len(serialized)} block(s) to {sender} (start_index={effective_start}, total_length={total_len})")
            self.send(sender, {
                "type": "BLOCKS",
                "from": self.address,
                "start_index": effective_start,
                "blocks": serialized,
                "total_chain_length": total_len
            })

        # ------------------- BLOCKS (Exercise 4 & 5) -------------------
        elif t == "BLOCKS":
            sender = tuple(msg["from"])
            start_index = msg.get("start_index", 0)
            raw_blocks = msg.get("blocks", [])
            total_chain_len = msg.get("total_chain_length", len(raw_blocks))
            new_blocks = [dict_to_block(b) for b in raw_blocks]

            print(f"[<] Received {len(new_blocks)} block(s) from {sender} (start_index={start_index})")
            updated = False

            with self.lock:
                if start_index == len(self.chain) and len(new_blocks) > 0:
                    # Delta update: append missing blocks directly
                    candidate_blocks = list(self.chain) + new_blocks
                    candidate_chain = JChain.Blockchain(candidate_blocks)
                    if self.is_chain_valid(candidate_chain):
                        self.chain = candidate_chain
                        updated = True
                        print(f"[★] Clever transmission: received {len(new_blocks)} block(s) instead of full {total_chain_len} blocks!")
                        print(f"[✓] Chain updated to length {len(self.chain)} (hash: {self.chain.hash()})")
                    else:
                        print("[-] Delta blocks failed validity check.")

                elif start_index == 0 and len(new_blocks) > len(self.chain):
                    # Full chain update (initial sync or reorganization)
                    candidate_chain = JChain.Blockchain(new_blocks)
                    if self.is_chain_valid(candidate_chain):
                        self.chain = candidate_chain
                        updated = True
                        print(f"[✓] Adopted full chain: length {len(self.chain)} (hash: {self.chain.hash()})")
                    else:
                        print("[-] Full chain failed validity check.")

            if updated:
                # Exercise 5: Whenever the current chain is updated, broadcast length and terminal hash!
                self.broadcast_chain_update()

        # ------------------- TEST -------------------
        elif t == "TEST":
            print(f"[+] Test message from {msg.get('from')}: {msg.get('content')}")


---
## Exercise 1: Demonstrating `max_peers`

**Requirement**: *To make our network more realistic: implement a `max_peers` parameter and only allow connection to up to this number of peers.*

In the test below:
1. We initialize a node with `max_peers = 2`.
2. We provide 3 initial peers: the node accepts only the first 2 and discards the 3rd.
3. We attempt to add a 4th peer via `add_peer()`: the connection is rejected because the peer table is at capacity (`2/2`).


In [4]:
print("=== EXERCISE 1 TEST: max_peers Enforcement ===")
test_node = Node('localhost', 30001, peers=[
    ('localhost', 30002),
    ('localhost', 30003),
    ('localhost', 30004)
], max_peers=2)

print(f"Configured max_peers: {test_node.max_peers}")
print(f"Current connected peers: {test_node.peers} (Count: {len(test_node.peers)})")
assert len(test_node.peers) == 2, "Peer count must not exceed max_peers!"

# Attempt to add another peer
attempt = test_node.add_peer(('localhost', 30005))
print(f"Attempting to add peer ('localhost', 30005): added = {attempt}")
assert not attempt, "Node must reject connections once max_peers is reached!"
assert len(test_node.peers) == 2

print("\n[SUCCESS] Exercise 1 verified: max_peers strictly enforced.")


=== EXERCISE 1 TEST: max_peers Enforcement ===
Configured max_peers: 2
Current connected peers: [('localhost', 30002), ('localhost', 30003)] (Count: 2)
[-] [max_peers=2] Limit reached. Cannot add peer ('localhost', 30005)
Attempting to add peer ('localhost', 30005): added = False

[SUCCESS] Exercise 1 verified: max_peers strictly enforced.


---
## Exercise 2: Detecting Unreachable Nodes (`check_alive_peers` / Ping)

**Requirement**: *Maybe some known nodes become unreachable: add a command to ask all known peers if they are still alive and update the list of (available) known peers accordingly.*

In the test below:
1. We launch three active nodes on separate ports (`Node 1`, `Node 2`, `Node 3`).
2. `Node 1` connects to both `Node 2` and `Node 3`.
3. We simulate a sudden crash/departure of `Node 2` by stopping its listener socket.
4. `Node 1` executes `check_alive_peers()`.
5. `Node 1` detects that `Node 2` is unreachable, prunes it from `self.peers`, and retains `Node 3`.


In [5]:
print("=== EXERCISE 2 TEST: Unreachable Nodes Liveness Check ===")
n1 = Node('localhost', 30011, max_peers=5)
n2 = Node('localhost', 30012, max_peers=5)
n3 = Node('localhost', 30013, max_peers=5)

n1.start()
n2.start()
n3.start()
time.sleep(0.1)

# Connect Node 1 to Node 2 and Node 3
n1.add_peer(n2.address)
n1.add_peer(n3.address)
print(f"Initial Node 1 peers: {n1.peers}")
assert len(n1.peers) == 2

# Simulate Node 2 going down
print("\n[Simulating crash] Shutting down Node 2...")
n2.stop()
time.sleep(0.2)

# Node 1 executes liveness check
alive, unreachable = n1.check_alive_peers(timeout=0.5)

print(f"\nAlive nodes: {alive}")
print(f"Unreachable nodes detected & removed: {unreachable}")

assert n2.address in unreachable, "Node 2 should have been detected as unreachable!"
assert n3.address in alive, "Node 3 should still be alive!"
assert len(n1.peers) == 1
assert n1.peers[0] == n3.address

print("\n[SUCCESS] Exercise 2 verified: unreachable nodes pruned successfully.")


=== EXERCISE 2 TEST: Unreachable Nodes Liveness Check ===
[+] Node listening on ('localhost', 30011) (max_peers=5)
[*] Asking all known peers for their chain lengths & terminal hashes...
[+] Node listening on ('localhost', 30012) (max_peers=5)
[*] Asking all known peers for their chain lengths & terminal hashes...
[+] Node listening on ('localhost', 30013) (max_peers=5)
[*] Asking all known peers for their chain lengths & terminal hashes...
[+] Added peer ('localhost', 30012) (1/5 connected)
[+] Added peer ('localhost', 30013) (2/5 connected)
Initial Node 1 peers: [('localhost', 30012), ('localhost', 30013)]

[Simulating crash] Shutting down Node 2...
[*] Node on 30012 stopped.
[*] Checking liveness of 2 known peer(s)...
    [X]  Peer ('localhost', 30012) is unreachable! Pruning from peers list.
    [OK] Peer ('localhost', 30013) is alive.
[+] Liveness check complete: 1 available, 1 removed.

Alive nodes: [('localhost', 30013)]
Unreachable nodes detected & removed: [('localhost', 30012

---
## Exercise 3: Chain Inquiry on Join (`request_chain_info`)

**Requirement**: *After a node joins the network, ask peers for the lengths and terminal hashes of their chains.*

In the test below:
1. `Node 1` creates two blocks in its local blockchain.
2. A newly created node (`Node 4`) starts with an empty chain.
3. Upon calling `node4.start()`, it announces itself (`HELLO`) and automatically triggers `request_chain_info()`, broadcasting `GET_CHAIN_INFO`.
4. `Node 1` replies with its chain length (`2`) and its terminal hash.


In [6]:
print("=== EXERCISE 3 TEST: Chain Inquiry on Join ===")
# Node 1 creates 2 blocks
n1.add_block("Tx 1: Genesis deposit Alice = 100")
n1.add_block("Tx 2: Transfer Alice -> Bob (25)")
print(f"Node 1 chain length: {len(n1.chain)}, terminal hash: {n1.chain.hash()}")
assert len(n1.chain) == 2

# New Node 4 starts with empty chain and knows Node 1
n4 = Node('localhost', 30014, peers=[n1.address], max_peers=5)
print(f"Node 4 initial chain length: {len(n4.chain)}")
assert len(n4.chain) == 0

# Node 4 joins the network -> automatically asks for chain lengths & terminal hashes!
print("\nStarting Node 4 (joining the network)...")
n4.start()
time.sleep(0.5)

# Verify Node 4 received chain info and synced to Node 1's chain
print(f"\nNode 4 chain length after inquiry & sync: {len(n4.chain)}")
print(f"Node 4 terminal hash: {n4.chain.hash()}")
assert len(n4.chain) == 2
assert n4.chain.hash() == n1.chain.hash()

print("\n[SUCCESS] Exercise 3 verified: chain inquiry automatically sent on join.")


=== EXERCISE 3 TEST: Chain Inquiry on Join ===
[+] New block added locally: length=1, hash=26235735
[!] Broadcasting CHAIN_INFO: length=1, terminal_hash=26235735
[+] New block added locally: length=2, hash=3509426853
[!] Broadcasting CHAIN_INFO: length=2, terminal_hash=3509426853
[i] CHAIN_INFO from ('localhost', 30011): length=1, hash=26235735
[*] Peer ('localhost', 30011) has a longer chain (1 > 0). Requesting update...
Node 1 chain length: 2, terminal hash: 3509426853
Node 4 initial chain length: 0

Starting Node 4 (joining the network)...
[+] Node listening on ('localhost', 30014) (max_peers=5)
[>] Transmitting 2 block(s) to ('localhost', 30013) (start_index=0, total_length=2)
[i] CHAIN_INFO from ('localhost', 30011): length=2, hash=3509426853
[*] Peer ('localhost', 30011) has a longer chain (2 > 0). Requesting update...
[>] Transmitting 2 block(s) to ('localhost', 30013) (start_index=0, total_length=2)
[<] Received 2 block(s) from ('localhost', 30011) (start_index=0)
[★] Clever tr

---
## Exercise 4: Clever Synchronization (Minimizing Transmitted Blocks)

**Requirement**: *Whenever information about a length and terminal hash is received, update the current version of the chain accordingly. (If you are clever, you will try to minimize the number of blocks transmitted.)*

### How the Clever Sync Works:
1. **Longest-Chain Rule**: Between two valid chains, a node always chooses the longest one (slide 3 of the lecture).
2. **Delta Transmission**: If Node A has $L_A$ blocks with terminal hash $H_A$, and receives a `CHAIN_INFO` from Node B with length $L_B > L_A$:
   - Node A sends `GET_BLOCKS(start_index=L_A, last_hash=H_A)`.
   - Node B verifies that its block at index $L_A - 1$ has hash equal to $H_A$ (common ancestor check).
   - Node B transmits **only blocks $L_A$ to $L_B-1$** (i.e. only $L_B - L_A$ blocks instead of the full $L_B$ chain!).
   - Node A receives only the new delta block(s) and appends them to its chain.
   - **Bandwidth savings**: If the chain has 1,000 blocks and 1 new block is added, only **1 block** is transmitted instead of all 1,001 blocks!

In the test below:
- `Node 1` has 2 blocks and adds a 3rd block.
- `Node 4` (which already has 2 blocks) receives `CHAIN_INFO(length=3)`.
- `Node 4` requests blocks starting at index `2`.
- `Node 1` transmits **exactly 1 block** instead of 3.
- `Node 4` updates its chain to length 3 with matching terminal hash.


In [7]:
print("=== EXERCISE 4 TEST: Clever Synchronization (Delta Sync) ===")
# Node 1 adds a 3rd block
print("Node 1 adds 3rd block (Mining new block)...")
n1.add_block("Tx 3: Transfer Bob -> Charlie (10)")

# Allow network messages to propagate
time.sleep(0.5)

print(f"\nNode 1 final chain length: {len(n1.chain)}, hash: {n1.chain.hash()}")
print(f"Node 4 final chain length: {len(n4.chain)}, hash: {n4.chain.hash()}")

assert len(n4.chain) == 3, "Node 4 should have updated its chain to length 3!"
assert n4.chain.hash() == n1.chain.hash(), "Node 4 and Node 1 terminal hashes must match!"

print("\n[SUCCESS] Exercise 4 verified: chain updated using minimal block transmission.")


=== EXERCISE 4 TEST: Clever Synchronization (Delta Sync) ===
Node 1 adds 3rd block (Mining new block)...
[+] New block added locally: length=3, hash=768086086
[!] Broadcasting CHAIN_INFO: length=3, terminal_hash=768086086
[i] CHAIN_INFO from ('localhost', 30011): length=3, hash=768086086
[*] Peer ('localhost', 30011) has a longer chain (3 > 2). Requesting update...
[*] Clever sync: asking only for missing blocks from index 2 (saving 2 blocks)!
[>] Transmitting 1 block(s) to ('localhost', 30013) (start_index=2, total_length=3)
[i] CHAIN_INFO from ('localhost', 30011): length=3, hash=768086086
[*] Peer ('localhost', 30011) has a longer chain (3 > 2). Requesting update...
[*] Clever sync: asking only for missing blocks from index 2 (saving 2 blocks)!
[<] Received 1 block(s) from ('localhost', 30011) (start_index=2)
[★] Clever transmission: received 1 block(s) instead of full 3 blocks!
[✓] Chain updated to length 3 (hash: 768086086)
[!] Broadcasting CHAIN_INFO: length=3, terminal_hash=7680

---
## Exercise 5: Broadcasting Chain Updates to the Network

**Requirement**: *Whenever the current chain is updated, the current length and terminal hash of the chain is broadcasted.*

### Mechanism:
Whenever `add_block()` is executed locally, or whenever new valid blocks are integrated into `self.chain`, `broadcast_chain_update()` is called to notify all connected peers:
```python
def broadcast_chain_update(self):
    self.broadcast({
        "type": "CHAIN_INFO",
        "from": self.address,
        "length": len(self.chain),
        "hash": self.chain.hash()
    })
```
This ensures implicit consensus propagates across the entire peer-to-peer network without centralized coordination.

In the test below:
1. We set up a network topology: `Node A` $\leftrightarrow$ `Node B` $\leftrightarrow$ `Node C`.
2. `Node A` adds a new block.
3. `Node A` broadcasts its update to `Node B`.
4. `Node B` syncs the new block and broadcasts the update to `Node C`.
5. `Node C` syncs the new block.
6. All three nodes reach consensus on the exact same chain length and terminal hash!


In [8]:
print("=== EXERCISE 5 TEST: Network-Wide Consensus Propagation ===")
# Setup 3 fresh nodes
nodeA = Node('localhost', 30021, max_peers=5)
nodeB = Node('localhost', 30022, max_peers=5)
nodeC = Node('localhost', 30023, max_peers=5)

nodeA.start()
nodeB.start()
nodeC.start()
time.sleep(0.1)

# Connect: A <-> B and B <-> C (Linear topology)
nodeA.add_peer(nodeB.address)
nodeB.add_peer(nodeA.address)
nodeB.add_peer(nodeC.address)
nodeC.add_peer(nodeB.address)
time.sleep(0.1)

# Node A creates a genesis block
print("\n>>> Node A mines Genesis Block...")
nodeA.add_block("Genesis: System initialization")
time.sleep(0.4)

# Node A mines a second block
print("\n>>> Node A mines Block 1...")
nodeA.add_block("Block 1: Transfer Alice -> Bob 50 coins")
time.sleep(0.6)

print("\n--- Checking Network Consensus State ---")
print(f"Node A: length={len(nodeA.chain)}, hash={nodeA.chain.hash()}")
print(f"Node B: length={len(nodeB.chain)}, hash={nodeB.chain.hash()}")
print(f"Node C: length={len(nodeC.chain)}, hash={nodeC.chain.hash()}")

assert len(nodeA.chain) == 2
assert len(nodeB.chain) == 2, "Node B must have received broadcast and updated!"
assert len(nodeC.chain) == 2, "Node C must have received broadcast via B and updated!"
assert nodeA.chain.hash() == nodeB.chain.hash() == nodeC.chain.hash(), "All nodes must agree on terminal hash!"

print("\n[SUCCESS] Exercise 5 verified: chain updates successfully broadcasted and propagated across network.")

# Clean shutdown of all test nodes
for n in [n1, n3, n4, nodeA, nodeB, nodeC]:
    n.stop()
print("\nAll test nodes safely stopped.")


=== EXERCISE 5 TEST: Network-Wide Consensus Propagation ===
[+] Node listening on ('localhost', 30021) (max_peers=5)
[*] Asking all known peers for their chain lengths & terminal hashes...
[+] Node listening on ('localhost', 30022) (max_peers=5)
[*] Asking all known peers for their chain lengths & terminal hashes...
[+] Node listening on ('localhost', 30023) (max_peers=5)
[*] Asking all known peers for their chain lengths & terminal hashes...
[+] Added peer ('localhost', 30022) (1/5 connected)
[+] Added peer ('localhost', 30021) (1/5 connected)
[+] Added peer ('localhost', 30023) (2/5 connected)
[+] Added peer ('localhost', 30022) (1/5 connected)

>>> Node A mines Genesis Block...
[+] New block added locally: length=1, hash=3619812173
[!] Broadcasting CHAIN_INFO: length=1, terminal_hash=3619812173
[i] CHAIN_INFO from ('localhost', 30021): length=1, hash=3619812173
[*] Peer ('localhost', 30021) has a longer chain (1 > 0). Requesting update...
[>] Transmitting 1 block(s) to ('localhost',